In [230]:
import numpy as np
import pandas as pd
import datetime as dt
from pathlib import Path
from __future__ import annotations
from collections.abc import Sequence
from typing import Hashable


DATA_DIR = Path("data")
FEATURE_DIR = Path("stored_features")
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

In [231]:
games_df = pd.read_csv(DATA_DIR / "international_football_games.csv")
draws_df = pd.read_csv(DATA_DIR / "shootouts.csv")
valid_teams_df = pd.read_csv(DATA_DIR / "valid_fifa_teams.csv")


In [232]:
print(games_df["tournament"].unique().tolist())

['Friendly', 'British Home Championship', 'Évence Coppée Trophy', 'Muratti Vase', 'Copa Lipton', 'Copa Newton', 'Copa Premio Honor Argentino', 'Olympic Games', 'Copa Premio Honor Uruguayo', 'Far Eastern Championship Games', 'Copa Roca', 'Copa América', 'Inter-Allied Games', 'Peace Cup', 'Open International Championship', 'Soccer Ashes', 'Copa Chevallier Boutell', 'Nordic Championship', 'Central European International Cup', 'Baltic Cup', 'Balkan Cup', 'Central American and Caribbean Games', 'FIFA World Cup', 'Copa Rio Branco', 'FIFA World Cup qualification', 'Bolivarian Games', 'CCCF Championship', 'NAFC Championship', 'Copa Oswaldo Cruz', 'Asian Games', 'Pan American Championship', 'Copa del Pacífico', "Copa Bernardo O'Higgins", 'AFC Asian Cup qualification', 'Atlantic Cup', 'AFC Asian Cup', 'African Cup of Nations', 'Copa Paz del Chaco', 'Merdeka Tournament', 'UEFA Euro qualification', 'Southeast Asian Peninsular Games', 'African Friendship Games', 'UEFA Euro', 'Windward Islands Tourn

In [233]:
home_teams = games_df["home_team"].unique().tolist()
away_teams = games_df["away_team"].unique().tolist()
all_teams = set(home_teams + away_teams)
for team in valid_teams_df["team"]:
    if team not in all_teams:
        print(f"{team} not in any teams")

In [234]:
teams = set(valid_teams_df["team"].dropna().unique())
print(games_df["date"].min(), games_df["date"].max(), games_df.shape[0])

valid_team_mask = (
    games_df["home_team"].isin(teams)
    & games_df["away_team"].isin(teams)
)
removed_games = (~valid_team_mask).sum()
games_df = games_df.loc[valid_team_mask].copy()

print(
    games_df["date"].min(),
    games_df["date"].max(),
    games_df.shape,
    f"removed {removed_games} games with invalid teams",
)

1872-11-30 2026-07-18 49508
1872-11-30 2026-07-18 (45502, 9) removed 4006 games with invalid teams


In [235]:
display(games_df.head(10))

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False
5,1876-03-25,Scotland,Wales,4,0,Friendly,Glasgow,Scotland,False
6,1877-03-03,England,Scotland,1,3,Friendly,London,England,False
7,1877-03-05,Wales,Scotland,0,2,Friendly,Wrexham,Wales,False
8,1878-03-02,Scotland,England,7,2,Friendly,Glasgow,Scotland,False
9,1878-03-23,Scotland,Wales,9,0,Friendly,Glasgow,Scotland,False


In [236]:
winner = None
games_df=games_df.merge(draws_df, how='left', left_on=['home_team', 'away_team', 'date'], right_on=['home_team', 'away_team', 'date']).drop(columns=['first_shooter'])
games_df["date"] = pd.to_datetime(games_df["date"])

In [237]:
display(games_df[games_df['home_team'].isin(['Argentina', 'France']) & (games_df["date"] > dt.datetime(2022, 12, 1))  ])

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,winner
41958,2022-12-03,Argentina,Australia,2,1,FIFA World Cup,Al Rayyan,Qatar,True,NaN
41959,2022-12-04,France,Poland,3,1,FIFA World Cup,Doha,Qatar,True,NaN
41971,2022-12-13,Argentina,Croatia,3,0,FIFA World Cup,Lusail,Qatar,True,NaN
41972,2022-12-14,France,Morocco,2,0,FIFA World Cup,Al Khor,Qatar,True,NaN
41978,2022-12-18,Argentina,France,3,3,FIFA World Cup,Lusail,Qatar,True,Argentina
...,...,...,...,...,...,...,...,...,...,...
45492,2026-07-07,Argentina,Egypt,3,2,FIFA World Cup,Kansas City,United States,True,NaN
45495,2026-07-09,France,Morocco,2,0,FIFA World Cup,Foxborough,United States,True,NaN
45498,2026-07-12,Argentina,Switzerland,3,1,FIFA World Cup,Kansas City,United States,True,NaN
45499,2026-07-14,France,Spain,0,2,FIFA World Cup,Arlington,United States,True,NaN


In [238]:
games_df.loc[games_df["home_score"] > games_df["away_score"], "winner"] = games_df["home_team"]
games_df.loc[games_df["home_score"] < games_df["away_score"], "winner"] = games_df["away_team"]
games_df.loc[
    (games_df["home_score"] == games_df["away_score"]) &
    (games_df["winner"].isna()),
    "winner"
] = 'DRAW'

games_df["winner_code"] = np.select(
    [
        games_df["winner"] == games_df["home_team"],
        games_df["winner"] == games_df["away_team"],
        games_df["winner"] == "DRAW",
    ],
    [0, 2, 1],
    default=-1,
)
games_df["_match_occurrence"] = (
    games_df.groupby(
        ["date", "home_team", "away_team"],
        dropna=False,
    )
    .cumcount()
)

games_df["match_id"] = (
    games_df["date"].astype(str)
    + "_"
    + games_df["home_team"].astype(str).replace(" ", "_")
    + "_"
    + games_df["away_team"].astype(str).replace(" ", "_")
    + "_"
    + games_df["_match_occurrence"].astype(str)
)

games_df.drop(columns="_match_occurrence", inplace=True)

In [239]:
GAMES_VALID_PATH = FEATURE_DIR / "_games_valid_teams.csv"

In [240]:
games_df.to_csv(GAMES_VALID_PATH, index=False)

In [241]:
games_df = pd.read_csv(GAMES_VALID_PATH, parse_dates=["date"])

In [242]:
print(games_df.iloc[24590:24600])

            date      home_team     away_team  home_score  away_score  \
24590 2003-05-22   South Africa       England           1           2   
24591 2003-05-24       Botswana        Malawi           1           1   
24592 2003-05-24          Sudan         Kenya           1           1   
24593 2003-05-25        Jamaica       Nigeria           3           2   
24594 2003-05-25        Lesotho      Eswatini           2           1   
24595 2003-05-25           Togo         Benin           3           1   
24596 2003-05-26  United States         Wales           2           0   
24597 2003-05-27       Scotland   New Zealand           1           1   
24598 2003-05-29        Algeria  Burkina Faso           0           1   
24599 2003-05-30        Nigeria         Ghana           3           1   

       tournament       city        country  neutral         winner  \
24590    Friendly     Durban   South Africa    False        England   
24591  COSAFA Cup   Gaborone       Botswana    False  

In [243]:
home = games_df[["date", "home_team", "away_team", "home_score", "away_score", "winner", "neutral", "match_id", "tournament"]].rename(
    columns={"home_team": "team", "away_team": "opponent", "home_score": "score", "away_score": "opponent_score", "winner": "winner", "match_id": "match_id", "tournament": "tournament"}
)
home['is_home'] = 1

away = games_df[["date", "away_team", "home_team", "home_score", "away_score", "winner", "neutral", "match_id", "tournament"]].rename(
    columns={"away_team": "team", "home_team": "opponent", "away_score": "score", "home_score": "opponent_score", "winner": "winner", "match_id": "match_id", "tournament": "tournament"}
)
away['is_home'] = 0
team_features_per_date = pd.concat([home, away]).sort_values(["date", "match_id"]).reset_index(drop=True)
team_features_per_date["date"] = pd.to_datetime(
    team_features_per_date["date"]
)

In [244]:
team_features_per_date.tail(80)

,date,team,opponent,score,opponent_score,winner,neutral,match_id,tournament,is_home
90924,2026-06-26,Norway,France,1,4,France,True,2026-06-26_Norway_France_0,FIFA World Cup,1
90925,2026-06-26,France,Norway,4,1,France,True,2026-06-26_Norway_France_0,FIFA World Cup,0
90926,2026-06-26,Senegal,Iraq,5,0,Senegal,True,2026-06-26_Senegal_Iraq_0,FIFA World Cup,1
90927,2026-06-26,Iraq,Senegal,0,5,Senegal,True,2026-06-26_Senegal_Iraq_0,FIFA World Cup,0
90928,2026-06-26,Tunisia,Netherlands,1,3,Netherlands,True,2026-06-26_Tunisia_Netherlands_0,FIFA World Cup,1
...,...,...,...,...,...,...,...,...,...,...
90999,2026-07-14,Spain,France,2,0,Spain,True,2026-07-14_France_Spain_0,FIFA World Cup,0
91000,2026-07-15,England,Argentina,1,2,Argentina,True,2026-07-15_England_Argentina_0,FIFA World Cup,1
91001,2026-07-15,Argentina,England,2,1,Argentina,True,2026-07-15_England_Argentina_0,FIFA World Cup,0
91002,2026-07-18,France,England,4,6,England,True,2026-07-18_France_England_0,FIFA World Cup,1


In [245]:
display(team_features_per_date[team_features_per_date['team'].isin(['Argentina', 'France']) & (team_features_per_date["date"] > dt.datetime(2022, 12, 1))  ])

,date,team,opponent,score,opponent_score,winner,neutral,match_id,tournament,is_home
83914,2022-12-03,Argentina,Australia,2,1,Argentina,True,2022-12-03_Argentina_Australia_0,FIFA World Cup,1
83920,2022-12-04,France,Poland,3,1,France,True,2022-12-04_France_Poland_0,FIFA World Cup,1
83935,2022-12-09,Argentina,Netherlands,2,2,Argentina,True,2022-12-09_Netherlands_Argentina_0,FIFA World Cup,0
83937,2022-12-10,France,England,2,1,France,True,2022-12-10_England_France_0,FIFA World Cup,0
83942,2022-12-13,Argentina,Croatia,3,0,Argentina,True,2022-12-13_Argentina_Croatia_0,FIFA World Cup,1
...,...,...,...,...,...,...,...,...,...,...
90990,2026-07-09,France,Morocco,2,0,France,True,2026-07-09_France_Morocco_0,FIFA World Cup,1
90996,2026-07-12,Argentina,Switzerland,3,1,Argentina,True,2026-07-12_Argentina_Switzerland_0,FIFA World Cup,1
90998,2026-07-14,France,Spain,0,2,Spain,True,2026-07-14_France_Spain_0,FIFA World Cup,1
91001,2026-07-15,Argentina,England,2,1,Argentina,True,2026-07-15_England_Argentina_0,FIFA World Cup,0


In [246]:
team_features_per_date["points"] = np.select(
    [
        team_features_per_date["team"] == team_features_per_date["winner"],
        team_features_per_date["winner"] == "DRAW",
    ],
    [3, 1],
    default=0,
)


In [247]:
team_features_per_date["_goal_difference"] = (
    team_features_per_date["score"]
    - team_features_per_date["opponent_score"]
)
team_features_per_date["_total_goals"] = (
    team_features_per_date["score"]
    + team_features_per_date["opponent_score"]
)
team_features_per_date["_clean_sheet"] = (
    team_features_per_date["opponent_score"].eq(0)
)
team_features_per_date["_failed_to_score"] = (
    team_features_per_date["score"].eq(0)
)
team_features_per_date["_over_2_5"] = (
    team_features_per_date["_total_goals"].gt(2.5)
)
team_features_per_date["_over_3_5"] = (
    team_features_per_date["_total_goals"].gt(3.5)
)
team_features_per_date["_under_1_5"] = (
    team_features_per_date["_total_goals"].lt(1.5)
)

team_features_per_date["ppg_last_5"] = (
    team_features_per_date
    .groupby("team")["points"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
team_features_per_date["ppg_last_10"] = (
    team_features_per_date
    .groupby("team")["points"]
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)


team_features_per_date["avg_goals_scored_last_5"] = (
    team_features_per_date
    .groupby("team")["score"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
team_features_per_date["avg_goals_conceded_last_5"] = (
    team_features_per_date
    .groupby("team")["opponent_score"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
team_features_per_date["clean_sheets_rate_last_5"] = (
    team_features_per_date
    .groupby("team")["_clean_sheet"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
team_features_per_date["avg_goal_difference_last_5"] = (
    team_features_per_date
    .groupby("team")["_goal_difference"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
team_features_per_date["avg_goal_difference_last_10"] = (
    team_features_per_date
    .groupby("team")["_goal_difference"]
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)
team_features_per_date["btts"] = (
    (team_features_per_date["score"] > 0)
    & (team_features_per_date["opponent_score"] > 0)
)
team_features_per_date["btts_rate_last_5"] = (
    team_features_per_date
    .groupby("team")["btts"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
team_features_per_date["failed_score_rate_last_5"] = (
    team_features_per_date
    .groupby("team")["_failed_to_score"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
team_features_per_date["over_2_5_rate_last_5"] = (
    team_features_per_date
    .groupby("team")["_over_2_5"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
team_features_per_date["over_3_5_rate_last_5"] = (
    team_features_per_date
    .groupby("team")["_over_3_5"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
team_features_per_date["under_1_5_rate_last_5"] = (
    team_features_per_date
    .groupby("team")["_under_1_5"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
team_features_per_date["days_since_last_game"] = (
    team_features_per_date
    .groupby("team")["date"]
    .diff()
    .dt.days
    .fillna(0)
)

team_features_per_date["goal_difference_trend"] = (
    team_features_per_date["avg_goal_difference_last_5"]
    - team_features_per_date["avg_goal_difference_last_10"]
)
team_features_per_date["momentum"] = (
    team_features_per_date["ppg_last_5"]
    - team_features_per_date["ppg_last_10"]
)

team_features_per_date.drop(
    columns=[
        "_goal_difference",
        "_total_goals",
        "_clean_sheet",
        "_failed_to_score",
        "_over_2_5",
        "_over_3_5",
        "_under_1_5",
        "btts"
    ],
    inplace=True,
)


In [248]:
team_features_per_date.columns

Index(['date', 'team', 'opponent', 'score', 'opponent_score', 'winner',
       'neutral', 'match_id', 'tournament', 'is_home', 'points', 'ppg_last_5',
       'ppg_last_10', 'avg_goals_scored_last_5', 'avg_goals_conceded_last_5',
       'clean_sheets_rate_last_5', 'avg_goal_difference_last_5',
       'avg_goal_difference_last_10', 'btts_rate_last_5',
       'failed_score_rate_last_5', 'over_2_5_rate_last_5',
       'over_3_5_rate_last_5', 'under_1_5_rate_last_5', 'days_since_last_game',
       'goal_difference_trend', 'momentum'],
      dtype='str')

In [249]:
def add_pi_ratings(
    team_features_per_date: pd.DataFrame,
    params: Sequence[float],
    *,
    match_id_col: str | None = None,
    strict: bool = True,
) -> pd.DataFrame:
    """
    Add pre-match Pi ratings to a team-per-row dataframe.

    Each match must have exactly two rows:
        team A vs team B
        team B vs team A

    Parameters
    ----------
    team_features_per_date:
        Team-per-row match dataframe.

    params:
        Sequence containing:

        c:
            Controls the nonlinear conversion between rating differences
            and expected goal differences. Must be greater than zero.

        mu1:
            Primary learning rate. Controls how strongly the venue-specific
            rating changes after a match.

        mu2:
            Cross-context learning rate. A proportion of the primary rating
            change is transferred to the team's other venue ratings.

            For example, after a home match, the team's away and neutral
            ratings each receive:

                home_rating_change * mu2

    match_id_col:
        Optional column containing a unique match identifier.

        Using a match identifier is strongly recommended. When omitted,
        matches are paired using:

            date + unordered team pair + occurrence number

    strict:
        If True, raise an exception when malformed match rows are found.
        If False, malformed matches are skipped and their Pi columns remain
        missing.

    Required columns
    ----------------
    date
    team
    opponent
    score
    opponent_score
    neutral

    Additionally, ``is_home`` is required for non-neutral matches.

    Returns
    -------
    pd.DataFrame
        Original dataframe, sorted chronologically, with these additional
        pre-match columns:

        pi_home_rating
        pi_away_rating
        pi_neutral_rating
        pi_context_rating
        pi_opponent_context_rating
        pi_expected_gd
        pi_diff
        pi_context

    Notes
    -----
    All Pi columns represent values before the result of the corresponding
    match is incorporated. This makes them appropriate as predictive
    features without leaking the current match result.
    """

    # ------------------------------------------------------------------
    # Validation
    # ------------------------------------------------------------------
    required_columns = {
        "date",
        "team",
        "opponent",
        "score",
        "opponent_score",
        "neutral",
    }

    missing = required_columns.difference(team_features_per_date.columns)
    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}"
        )

    if len(params) != 3:
        raise ValueError("params must contain exactly (c, mu1, mu2).")

    c, mu1, mu2 = map(float, params)

    if not np.isfinite(c) or c <= 0:
        raise ValueError("c must be a finite number greater than zero.")

    if not np.isfinite(mu1) or mu1 < 0:
        raise ValueError("mu1 must be a finite, non-negative number.")

    if not np.isfinite(mu2) or not 0 <= mu2 <= 1:
        raise ValueError("mu2 must be between 0 and 1.")

    df = team_features_per_date.copy()

    if df.empty:
        for column in [
            "pi_home_rating",
            "pi_away_rating",
            "pi_neutral_rating",
            "pi_context_rating",
            "pi_opponent_context_rating",
            "pi_expected_gd",
            "pi_diff",
            "pi_context",
        ]:
            df[column] = pd.Series(dtype="float64" if column != "pi_context" else "object")
        return df

    df["_pi_original_order"] = np.arange(len(df))

    # Stable ordering is important when several matches share a date.
    sort_columns = ["date"]
    if match_id_col is not None:
        if match_id_col not in df.columns:
            raise ValueError(
                f"match_id_col={match_id_col!r} is not present in the dataframe."
            )
        sort_columns.append(match_id_col)

    sort_columns.append("_pi_original_order")

    df = (
        df.sort_values(sort_columns, kind="stable")
        .reset_index(drop=True)
    )

    df["neutral"] = df["neutral"].astype(bool)

    has_non_neutral_matches = (~df["neutral"]).any()
    if has_non_neutral_matches and "is_home" not in df.columns:
        raise ValueError(
            "'is_home' is required because the dataframe contains "
            "non-neutral matches."
        )

    if "is_home" in df.columns:
        # Preserve missing values so malformed rows can be detected.
        df["is_home"] = df["is_home"].astype("boolean")

    # ------------------------------------------------------------------
    # Construct match groups
    # ------------------------------------------------------------------
    if match_id_col is not None:
        if df[match_id_col].isna().any():
            raise ValueError(f"{match_id_col!r} contains missing values.")

        match_groups = list(
            df.groupby(match_id_col, sort=False, dropna=False).groups.values()
        )

    else:
        # A match_id is preferable, but this handles ordinary mirrored rows.
        #
        # The occurrence counter lets the same two teams play more than once
        # on the same date, provided both row directions occur in the same
        # order.
        df["_pi_pair"] = [
            tuple(sorted((team, opponent), key=lambda value: str(value)))
            for team, opponent in zip(df["team"], df["opponent"])
        ]

        df["_pi_pair_occurrence"] = (
            df.groupby(
                ["date", "team", "opponent"],
                sort=False,
                dropna=False,
            )
            .cumcount()
        )

        match_groups = list(
            df.groupby(
                ["date", "_pi_pair", "_pi_pair_occurrence"],
                sort=False,
                dropna=False,
            ).groups.values()
        )

    # ------------------------------------------------------------------
    # Rating storage and helper functions
    # ------------------------------------------------------------------
    contexts = ("home", "away", "neutral")

    teams = pd.unique(
        pd.concat(
            [df["team"], df["opponent"]],
            ignore_index=True,
        )
    )

    ratings: dict[Hashable, dict[str, float]] = {
        team: {context: 0.0 for context in contexts}
        for team in teams
    }

    def signed_goal_value(rating: float) -> float:
        """
        Convert one Pi rating into its signed expected-goal contribution.
        """
        return float(
            np.sign(rating) * (10 ** (abs(rating) / c) - 1)
        )

    def expected_goal_difference(
        team_rating: float,
        opponent_rating: float,
    ) -> float:
        """
        Expected goal difference from the two context-specific ratings.
        """
        return (
            signed_goal_value(team_rating)
            - signed_goal_value(opponent_rating)
        )

    def calculate_rating_change(
        observed_gd: float,
        expected_gd: float,
    ) -> float:
        """
        Calculate the primary Pi rating change for the first team.

        The second team's change is the negative of this value.
        """
        residual = observed_gd - expected_gd

        if np.isclose(residual, 0.0):
            return 0.0

        weighted_residual = (
            np.sign(residual)
            * c
            * np.log10(1 + abs(residual))
        )

        return float(mu1 * weighted_residual)

    def update_team_ratings(
        team: Hashable,
        primary_context: str,
        primary_change: float,
    ) -> None:
        """
        Update the primary context and transfer part of the change to the
        team's two secondary contexts.
        """
        ratings[team][primary_context] += primary_change

        secondary_change = primary_change * mu2

        for context in contexts:
            if context != primary_context:
                ratings[team][context] += secondary_change

    def validation_error(message: str) -> bool:
        """
        Raise in strict mode; otherwise signal that the match should be
        skipped.
        """
        if strict:
            raise ValueError(message)
        return False

    # Preallocate result columns so skipped matches remain NaN.
    numeric_output_columns = [
        "pi_home_rating",
        "pi_away_rating",
        "pi_neutral_rating",
        "pi_context_rating",
        "pi_opponent_context_rating",
        "pi_expected_gd",
        "pi_diff",
    ]

    for column in numeric_output_columns:
        df[column] = np.nan

    df["pi_context"] = pd.Series(pd.NA, index=df.index, dtype="string")

    # ------------------------------------------------------------------
    # Process matches chronologically
    # ------------------------------------------------------------------
    for group_indices in match_groups:
        indices = list(group_indices)

        if len(indices) != 2:
            validation_error(
                "Each match must contain exactly two rows. "
                f"Found {len(indices)} rows at dataframe indices {indices}."
            )
            continue

        first_idx, second_idx = indices
        first = df.loc[first_idx]
        second = df.loc[second_idx]

        # Confirm that rows mirror one another.
        if not (
            first["team"] == second["opponent"]
            and first["opponent"] == second["team"]
        ):
            validation_error(
                "Match rows are not mirrored correctly at indices "
                f"{first_idx} and {second_idx}."
            )
            continue

        if bool(first["neutral"]) != bool(second["neutral"]):
            validation_error(
                "The two rows of a match disagree on the neutral flag at "
                f"indices {first_idx} and {second_idx}."
            )
            continue

        score_values = [
            first["score"],
            first["opponent_score"],
            second["score"],
            second["opponent_score"],
        ]

        if any(pd.isna(value) for value in score_values):
            validation_error(
                f"Missing score value at indices {first_idx} and {second_idx}."
            )
            continue

        scores_are_mirrored = (
            np.isclose(float(first["score"]), float(second["opponent_score"]))
            and np.isclose(
                float(first["opponent_score"]),
                float(second["score"]),
            )
        )

        if not scores_are_mirrored:
            validation_error(
                "Score columns are not mirrored correctly at indices "
                f"{first_idx} and {second_idx}."
            )
            continue

        is_neutral = bool(first["neutral"])

        if is_neutral:
            team_one_idx = first_idx
            team_two_idx = second_idx
            team_one = first["team"]
            team_two = second["team"]

            team_one_context = "neutral"
            team_two_context = "neutral"

        else:
            first_is_home = first["is_home"]
            second_is_home = second["is_home"]

            if pd.isna(first_is_home) or pd.isna(second_is_home):
                validation_error(
                    "Non-neutral match has a missing is_home value at "
                    f"indices {first_idx} and {second_idx}."
                )
                continue

            if bool(first_is_home) == bool(second_is_home):
                validation_error(
                    "A non-neutral match must have exactly one home row at "
                    f"indices {first_idx} and {second_idx}."
                )
                continue

            if bool(first_is_home):
                team_one_idx = first_idx
                team_two_idx = second_idx
            else:
                team_one_idx = second_idx
                team_two_idx = first_idx

            team_one = df.at[team_one_idx, "team"]
            team_two = df.at[team_two_idx, "team"]

            team_one_context = "home"
            team_two_context = "away"

        # Snapshot all pre-match ratings before making any update.
        team_one_ratings = ratings[team_one].copy()
        team_two_ratings = ratings[team_two].copy()

        team_one_context_rating = team_one_ratings[team_one_context]
        team_two_context_rating = team_two_ratings[team_two_context]

        expected_gd = expected_goal_difference(
            team_one_context_rating,
            team_two_context_rating,
        )

        observed_gd = float(
            df.at[team_one_idx, "score"]
            - df.at[team_one_idx, "opponent_score"]
        )

        team_one_change = calculate_rating_change(
            observed_gd,
            expected_gd,
        )
        team_two_change = -team_one_change

        # --------------------------------------------------------------
        # Store pre-match features for team one
        # --------------------------------------------------------------
        df.loc[team_one_idx, numeric_output_columns] = [
            team_one_ratings["home"],
            team_one_ratings["away"],
            team_one_ratings["neutral"],
            team_one_context_rating,
            team_two_context_rating,
            expected_gd,
            team_one_context_rating - team_two_context_rating,
        ]
        df.at[team_one_idx, "pi_context"] = team_one_context

        # --------------------------------------------------------------
        # Store pre-match features for team two
        # --------------------------------------------------------------
        df.loc[team_two_idx, numeric_output_columns] = [
            team_two_ratings["home"],
            team_two_ratings["away"],
            team_two_ratings["neutral"],
            team_two_context_rating,
            team_one_context_rating,
            -expected_gd,
            team_two_context_rating - team_one_context_rating,
        ]
        df.at[team_two_idx, "pi_context"] = team_two_context

        # Update only after both rows receive their pre-match features.
        update_team_ratings(
            team_one,
            team_one_context,
            team_one_change,
        )
        update_team_ratings(
            team_two,
            team_two_context,
            team_two_change,
        )

    # Remove internal helper columns.
    internal_columns = [
        "_pi_original_order",
        "_pi_pair",
        "_pi_pair_occurrence",
    ]

    df = df.drop(
        columns=[
            column
            for column in internal_columns
            if column in df.columns
        ]
    )

    return df

In [250]:
c, mu1, mu2 = 25, 0.20, 0.10
params = (
    c,    # c
    mu1,   # mu1
    mu2    # mu2
)

df = add_pi_ratings(team_features_per_date, params, strict=False)


PI_PARAMETER_TAG = f"c_{c:g}_mu1_{mu1:.2f}_mu2_{mu2:.2f}"
TEAM_FEATURES_PATH = FEATURE_DIR / f"team_features_{PI_PARAMETER_TAG}.csv"
MATCH_FEATURES_PATH = FEATURE_DIR / f"match_features_{PI_PARAMETER_TAG}.csv"

In [251]:
df.to_csv(TEAM_FEATURES_PATH, index=False)


In [252]:
team_features = pd.read_csv(TEAM_FEATURES_PATH, parse_dates=["date"])


In [253]:
df = team_features.copy()
home_ = df.add_prefix("home_")
away_ = df.add_prefix("away_")
home_.drop(columns=[  "home_opponent_score", "home_is_home", "home_points","home_match_id"], inplace=True)
away_.drop(columns=[ "away_opponent_score", "away_winner", "away_is_home", "away_points","away_neutral", "away_tournament"], inplace=True)

In [254]:
matches = home_.merge(
    away_,
    left_on=["home_date", "home_team", "home_opponent"],
    right_on=["away_date", "away_opponent", "away_team"],
    how="inner",
)
matches = matches[
    matches["home_team"] < matches["away_team"]
].reset_index(drop=True)
matches.drop(columns=["away_date", "away_opponent", "away_team"], inplace=True)
matches = matches.rename(columns={"home_opponent": "away_team", "home_date": "date", 'away_match_id':'match_id', "home_tournament": "tournament", "home_winner": "winner"})

In [255]:
# matches.loc[
#     matches["home_score"] > matches["away_score"], "winner"
# ] = matches["home_team"]
# matches.loc[
#     matches["home_score"] < matches["away_score"], "winner"
# ] = matches["away_team"]
# matches.loc[
#     (matches["home_score"] == matches["away_score"])
#     & matches["winner"].isna(),
#     "winner",
# ] = "DRAW"

matches["winner_code"] = np.select(
    [
        matches["winner"] == matches["home_team"],
        matches["winner"] == matches["away_team"],
        matches["winner"] == "DRAW",
    ],
    [0, 2, 1],
    default=-1,
)


In [256]:
matches = matches[["match_id",'date', 'home_team', 'away_team', 'home_score', 'away_score','winner','winner_code','tournament','home_neutral',
                   
       'home_ppg_last_5', 'home_ppg_last_10', 'home_avg_goals_scored_last_5',
       'home_avg_goals_conceded_last_5', 'home_clean_sheets_rate_last_5',
       'home_avg_goal_difference_last_5', 'home_avg_goal_difference_last_10',
        'home_btts_rate_last_5', 'home_failed_score_rate_last_5',
       'home_over_2_5_rate_last_5', 'home_over_3_5_rate_last_5',
       'home_under_1_5_rate_last_5', 'home_days_since_last_game',
       'home_goal_difference_trend', 'home_momentum', 'home_pi_home_rating',
       'home_pi_away_rating', 'home_pi_expected_gd', 'home_pi_diff',
       
       'away_ppg_last_5', 'away_ppg_last_10',
       'away_avg_goals_scored_last_5', 'away_avg_goals_conceded_last_5',
       'away_clean_sheets_rate_last_5', 'away_avg_goal_difference_last_5',
       'away_avg_goal_difference_last_10',
       'away_btts_rate_last_5', 'away_failed_score_rate_last_5',
       'away_over_2_5_rate_last_5', 'away_over_3_5_rate_last_5',
       'away_under_1_5_rate_last_5', 'away_days_since_last_game',
       'away_goal_difference_trend', 'away_momentum', 'away_pi_home_rating',
       'away_pi_away_rating', 'away_pi_expected_gd', 'away_pi_diff']]

In [257]:
matches.loc[matches["home_neutral"] == 1, "pi_rating_diff"] = matches["home_pi_away_rating"] - matches["away_pi_away_rating"]
matches.loc[matches["home_neutral"] == 0, "pi_rating_diff"] = matches["home_pi_home_rating"] - matches["away_pi_away_rating"]

matches["diff_ppg_last_5"] = matches["home_ppg_last_5"] - matches["away_ppg_last_5"]
matches["diff_goal_difference"] = matches["home_avg_goal_difference_last_5"] - matches["away_avg_goal_difference_last_5"]
matches["diff_btts_rate"] = matches["home_btts_rate_last_5"] - matches["away_btts_rate_last_5"]
matches["diff_avg_goals_scored_last_5"] = matches["home_avg_goals_scored_last_5"] - matches["away_avg_goals_scored_last_5"]
matches["diff_avg_goals_conceded_last_5"] = matches["away_avg_goals_conceded_last_5"] - matches["home_avg_goals_conceded_last_5"]
matches["diff_clean_sheets_rate"] = matches["home_clean_sheets_rate_last_5"] - matches["away_clean_sheets_rate_last_5"]
matches["diff_over_2_5_rate"] = matches["home_over_2_5_rate_last_5"] - matches["away_over_2_5_rate_last_5"]
matches["diff_under_1_5_rate"] = matches["home_under_1_5_rate_last_5"] - matches["away_under_1_5_rate_last_5"]
matches["diff_rests_days"] = matches["home_days_since_last_game"] - matches["away_days_since_last_game"]
matches["expected_gd_edge"] = matches["home_pi_expected_gd"] - matches["away_pi_expected_gd"]
matches["diff_momentum"] = matches["home_momentum"] - matches["away_momentum"]


matches["home_pi_momentum"] =  matches["home_pi_home_rating"] * matches["home_momentum"]
matches["away_pi_momentum"] = matches["away_pi_away_rating"] * matches["away_momentum"]
matches["home_weighted_avg_ppg"] = 0.7* matches["home_ppg_last_5"] + 0.3 * matches["home_ppg_last_10"]
matches["away_weighted_avg_ppg"] = 0.7* matches["away_ppg_last_5"] + 0.3 * matches["away_ppg_last_10"]
matches["home_goal_to_concede_rate"] = (
    matches["home_avg_goals_scored_last_5"]
    .div(matches["home_avg_goals_conceded_last_5"])
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

matches["away_goal_to_concede_rate"] = (
    matches["away_avg_goals_scored_last_5"]
    .div(matches["away_avg_goals_conceded_last_5"])
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)
#matches["home_weighted_avg_goals_scored"] = 0.7* matches["home_avg_goals_scored_last_5"] + 0.3 * matches["home_avg_goals_scored_last_10"]
#matches["away_weighted_avg_goals_scored"] = 0.7* matches["away_avg_goals_scored_last_5"] + 0.3 * matches["away_avg_goals_scored_last_10"]
#matches["home_weighted_avg_goals_conceded"] = 0.7* matches["home_avg_goals_conceded_last_5"] + 0.3 * matches["home_avg_goals_conceded_last_10"]
#matches["away_weighted_avg_goals_conceded"] = 0.7* matches["away_avg_goals_conceded_last_5"] + 0.3 * matches["away_avg_goals_conceded_last_10"]

# matches.loc[matches["winner_code"] != 0, "can_draw"] = 0
# matches.loc[matches["winner_code"] == 0, "can_draw"] = 1



In [258]:
matches.tail(60)

,match_id,date,home_team,away_team,home_score,away_score,winner,winner_code,tournament,home_neutral,...,diff_under_1_5_rate,diff_rests_days,expected_gd_edge,diff_momentum,home_pi_momentum,away_pi_momentum,home_weighted_avg_ppg,away_weighted_avg_ppg,home_goal_to_concede_rate,away_goal_to_concede_rate
45474,2026-06-21_Belgium_Iran_0,2026-06-21,Belgium,Iran,0,0,DRAW,1,FIFA World Cup,True,...,0.0,0.0,-0.970324,-1.000000e-01,0.000000,0.840886,2.20,1.97,3.500000,2.600000
45475,2026-06-21_New Zealand_Egypt_0,2026-06-21,Egypt,New Zealand,3,1,Egypt,0,FIFA World Cup,True,...,0.2,0.0,1.626356,-2.000000e-01,1.126014,3.876777,1.57,0.71,2.333333,0.600000
45476,2026-06-21_Spain_Saudi Arabia_0,2026-06-21,Saudi Arabia,Spain,0,4,Spain,2,FIFA World Cup,True,...,-0.2,0.0,-0.435190,3.000000e-01,-0.375021,-8.102962,1.03,1.92,1.200000,3.500000
45477,2026-06-21_Uruguay_Cape Verde_0,2026-06-21,Cape Verde,Uruguay,2,2,DRAW,1,FIFA World Cup,True,...,-0.2,0.0,-2.035060,9.000000e-01,0.000000,-11.153757,1.40,1.07,0.750000,0.428571
45478,2026-06-22_Argentina_Austria_0,2026-06-22,Argentina,Austria,2,0,Argentina,0,FIFA World Cup,True,...,-0.4,0.0,4.956758,2.000000e-01,4.882111,0.990280,2.91,2.57,15.000000,3.666667
45479,2026-06-22_France_Iraq_0,2026-06-22,France,Iraq,3,0,France,0,FIFA World Cup,True,...,-0.2,0.0,7.656241,1.000000e-01,-1.311965,-2.095262,2.43,1.46,2.000000,0.625000
45480,2026-06-22_Jordan_Algeria_0,2026-06-22,Algeria,Jordan,2,1,Algeria,0,FIFA World Cup,True,...,0.4,0.0,2.810646,8.000000e-01,-2.325408,-4.292170,2.06,0.70,4.000000,0.461538
45481,2026-06-22_Norway_Senegal_0,2026-06-22,Norway,Senegal,3,2,Norway,0,FIFA World Cup,True,...,0.0,0.0,-0.616839,0.000000e+00,-7.815949,-4.470291,1.75,1.55,1.800000,1.142857
45482,2026-06-23_Colombia_DR Congo_0,2026-06-23,Colombia,DR Congo,1,0,Colombia,0,FIFA World Cup,True,...,-0.4,0.0,1.631454,-2.000000e-01,-6.174423,-1.918326,1.92,1.66,1.428571,1.666667
45483,2026-06-23_England_Ghana_0,2026-06-23,England,Ghana,0,0,DRAW,1,FIFA World Cup,True,...,0.2,0.0,2.701057,0.000000e+00,-5.648441,-3.743987,2.15,0.95,2.250000,0.400000


In [259]:
matches.columns

Index(['match_id', 'date', 'home_team', 'away_team', 'home_score',
       'away_score', 'winner', 'winner_code', 'tournament', 'home_neutral',
       'home_ppg_last_5', 'home_ppg_last_10', 'home_avg_goals_scored_last_5',
       'home_avg_goals_conceded_last_5', 'home_clean_sheets_rate_last_5',
       'home_avg_goal_difference_last_5', 'home_avg_goal_difference_last_10',
       'home_btts_rate_last_5', 'home_failed_score_rate_last_5',
       'home_over_2_5_rate_last_5', 'home_over_3_5_rate_last_5',
       'home_under_1_5_rate_last_5', 'home_days_since_last_game',
       'home_goal_difference_trend', 'home_momentum', 'home_pi_home_rating',
       'home_pi_away_rating', 'home_pi_expected_gd', 'home_pi_diff',
       'away_ppg_last_5', 'away_ppg_last_10', 'away_avg_goals_scored_last_5',
       'away_avg_goals_conceded_last_5', 'away_clean_sheets_rate_last_5',
       'away_avg_goal_difference_last_5', 'away_avg_goal_difference_last_10',
       'away_btts_rate_last_5', 'away_failed_score_rate

In [260]:
# matches = matches.drop(columns=["home_btts", "away_btts", "home_ppg_last_10", "away_ppg_last_10",
#                                 "home_btts_rate_last_5", "away_btts_rate_last_5","home_goal_difference_trend", "away_goal_difference_trend","home_momentum", "away_momentum", "diff_clean_sheets_rate"])

In [261]:
matches.to_csv(MATCH_FEATURES_PATH, index=False)
print(f"Saved {len(matches):,} match rows to {MATCH_FEATURES_PATH}")

Saved 45,534 match rows to stored_features/match_features_c_25_mu1_0.20_mu2_0.10.csv


In [262]:
latest_team_ratings = (
    team_features
    .sort_values("date")
)

display(
    latest_team_ratings[
        [
            "date",
            "team",
            "pi_home_rating",
            "pi_away_rating",
            "pi_expected_gd",
            "pi_diff",
        ]
    ]
    .sort_values("pi_home_rating", ascending=False)
)
TEAM_RATING_RECORDS = FEATURE_DIR / "_team_ratings_record.csv"


,date,team,pi_home_rating,pi_away_rating,pi_expected_gd,pi_diff
45846,2001-04-14,Australia,25.179922,9.792130,9.913963,31.236419
46412,2001-06-24,Australia,24.902216,12.565133,8.838340,24.147326
46273,2001-06-03,Australia,24.598369,10.135838,0.259323,1.167545
46399,2001-06-20,Australia,24.557613,9.119101,-1.888847,-6.475004
46340,2001-06-09,Australia,24.489927,9.051414,0.634244,1.560101
...,...,...,...,...,...,...
46019,2001-05-04,Brunei,-23.856993,-14.265563,-3.012879,-17.048784
59753,2009-03-15,Macau,-24.004202,-20.258715,-2.537819,-5.415744
46062,2001-05-11,Brunei,-24.006105,-15.756685,-7.005186,-15.847442
45993,2001-04-27,Brunei,-24.145052,-17.146155,-3.768065,-16.279419


In [263]:
latest_team_ratings.to_csv(TEAM_RATING_RECORDS, index=False)
print(f"Saved {len(latest_team_ratings):,} team rating rows to {TEAM_RATING_RECORDS}")

Saved 91,004 team rating rows to stored_features/_team_ratings_record.csv
